In [2]:
import datetime
from decimal import Decimal

from simulator.engine import SimulationEngine
from simulator.wealth.cash_transfer import CashTransferSystem
from simulator.wealth.agents import Person, Company, RentalAgency, Landlord
from simulator.wealth.accounts import CashAccount
from simulator.wealth.contracts import EmploymentContract, TenancyAgreement

In [3]:
cash_transfer_system = CashTransferSystem(name="cash_transfer_system")

alice = Person(
    name="person:Alice",
    accounts={
        "cash": CashAccount(5000, "cash"),
    }
)

landlord = Landlord(
    name="landlord:Bob",
    accounts={
        "cash": CashAccount(0, "cash")
    }
)

company = Company(
    name="company:Tesla",
    accounts={
        "payroll": CashAccount(0, "payroll", allow_negative=True)
    },
    contracts=[
        EmploymentContract(
            employer="company:Tesla", 
            employee="person:Alice", 
            start_date=datetime.date(2026, 1, 3),
            annual_salary=Decimal(68400),
            employer_account_name="payroll",
            employee_account_name="cash",
            day_of_month=29,
            end_date=None,
            uid="full-time job"           
        ),
    ],
    cash_transfer_system="cash_transfer_system"
)

rental_agency = RentalAgency(
    name="rental_agency:HousingLtd",
    agreements=[
        TenancyAgreement(
            tenant="person:Alice",
            landlord="landlord:Bob",
            tenant_account_name="cash",
            landlord_account_name="cash",
            monthly_rent=Decimal(1250),
            payment_day=1,
            start_date=datetime.date(2025, 5, 1),
            end_date=None,
            uid="shared flat"
        )
    ],
    cash_transfer_system="cash_transfer_system"
)



engine = SimulationEngine(start_date=datetime.date(2026, 1, 1))
engine.register_many([cash_transfer_system, alice, company, landlord, rental_agency])  # Why does the type-checker complain?
engine.start()
engine.run_until(datetime.date(2026, 12, 1))

In [4]:
engine.event_log

[Event(date=datetime.date(2026, 1, 29), source='company:Tesla', target='company:Tesla', kind=<WealthEventKind.PAYROLL_DUE: 'payroll.due'>, payload=PayrollDuePayload(contract_uid='full-time job'), uid='4f8de095ebe348aeb09b66be306588aa', priority=20),
 Event(date=datetime.date(2026, 1, 29), source='company:Tesla', target='cash_transfer_system', kind=<WealthEventKind.CASH_TRANSFER: 'cash.transfer'>, payload=CashTransferPayload(source_account=AccountRef(module_name='company:Tesla', account_name='payroll'), target_account=AccountRef(module_name='person:Alice', account_name='cash'), amount=Decimal('5700')), uid='455d77851fb04f6b9fbb2c49d21e37fc', priority=30),
 Event(date=datetime.date(2026, 2, 1), source='rental_agency:HousingLtd', target='rental_agency:HousingLtd', kind=<WealthEventKind.RENT_DUE: 'rent.due'>, payload=RentDuePayload(agreement_uid='shared flat'), uid='d1f5726e198b4945b61b227992d8828e', priority=20),
 Event(date=datetime.date(2026, 2, 1), source='rental_agency:HousingLtd', ta

In [5]:
alice.net_worth

Decimal('53950')

In [ ]:
# event = engine.step()
# print(f"Event processed:\n{event}")
# print(f"date: {engine.today}, alice net-worth: {alice.net_worth}")

Event processed:
Event(date=datetime.date(2028, 3, 1), source='rental_agency:HousingLtd', target='rental_agency:HousingLtd', kind=<WealthEventKind.RENT_DUE: 'rent.due'>, payload=RentDuePayload(agreement_uid='shared flat'), uid='bb3e4809af8c447e870d18d5f694ede1', priority=20)
date: 2028-03-01, alice net-worth: 121950
